# Notebook 01 - Prompting and Steering Vectors

We are interested in methods for controlling language models in an interpretable way. Before digging into the interpretable part, we start with the simplest control method: changing the prompt. Prompting is useful, but it treats the model as a black box. We can observe that the answer changes, but we do not know where the relevant concept is represented inside the model.

From there, we introduce steering vectors: directions in activation space that can be added to a model's internal state to change its behavior. This idea is related to the classic word2vec "word algebra" intuition, where semantic relations appear as vector directions, such as `king - man + woman ≈ queen`. Here we ask whether a behavioral concept, like answering in Italian rather than English, can also be approximated by a direction in a model's residual stream.

This notebook is structured as follows:

1. Build a small English/Italian contrastive dataset.
2. Show black-box prompt control on English and Italian questions.
3. Capture residual-stream activations with PyTorch hooks.
4. Use 2D PCA to find layers where English and Italian activations separate.
5. Build a contrastive steering vector `mean(Italian) - mean(English)`.
6. Add or subtract that vector during generation and measure the output language.

The main lesson is that a concept can first be studied as a linear direction in activation space. Notebook 02 asks whether sparse autoencoder features give a more interpretable decomposition of that direction.

<p align="center">
  <img src="https://commons.wikimedia.org/wiki/Special:Redirect/file/Word%20vector%20illustration.jpg" alt="Word embedding analogy illustration" width="360"/>
</p>

*Intuition: word embeddings can encode semantic relations as directions, as in the classic `king - man + woman ≈ queen` example. Image source: [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Word_vector_illustration.jpg), CC BY-SA 4.0.*


## 0. Colab Setup

This cell installs the small package set needed in Colab. The repository is designed for a GPU runtime: `Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    packages = [
        "transformers>=5.10.1",
        "accelerate>=0.33.0",
        "huggingface_hub>=1.17.0",
        "pandas>=2.0.0",
        "matplotlib>=3.8.0",
        "tqdm>=4.66.0",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


In [ ]:
import re
import json
import random
import requests
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

DATA_DIR = Path("data")

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
BATCH_SIZE = 4
MAX_SENTENCES_PER_LANGUAGE = 50
MAX_NEW_TOKENS = 140
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float32

LANGUAGE_COLORS = {
    "english": "#4C72B0",
    "italian": "#55A868",
    "unknown": "#A0A0A0",
}
LANGUAGE_ORDER = ["english", "italian", "unknown"]
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
})

print(f"Device: {DEVICE}")


## 1. Dataset

This module loads the English/Italian sentences used to define the concept direction, plus held-out questions used for generation. The contrast is intentionally simple: language identity is easy to inspect, so it is a good first example of a model feature.


In [ ]:
languages = [
    "italian",
    "english",
]

file_types = [
    "sentences",
    "test_questions",
    "unseen_sentences",
]

base_url = "https://raw.githubusercontent.com/leobertolazzi/mech-interp-lab-lcl26/main/data"

DATA_DIR.mkdir(exist_ok=True)

for language in languages:
    for file_type in file_types:
        filename = f"{language}_{file_type}.json"
        url = f"{base_url}/{filename}"

        response = requests.get(url)
        response.raise_for_status()

        (DATA_DIR / filename).write_bytes(response.content)

print("All files downloaded.")

In [ ]:
def read_json_list(path: Path) -> list[str]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. In Colab, clone/download the repo or upload the data/ directory first."
        )
    data = json.loads(path.read_text())
    return data

english_sentences = read_json_list(DATA_DIR / "english_sentences.json")[:MAX_SENTENCES_PER_LANGUAGE]
italian_sentences = read_json_list(DATA_DIR / "italian_sentences.json")[:MAX_SENTENCES_PER_LANGUAGE]
english_test_questions = read_json_list(DATA_DIR / "english_test_questions.json")
italian_test_questions = read_json_list(DATA_DIR / "italian_test_questions.json")

print(f"English sentences: {len(english_sentences)}")
print(f"Italian sentences: {len(italian_sentences)}")
print(f"English test questions: {len(english_test_questions)}")
print(f"Italian test questions: {len(italian_test_questions)}")
print("\nExample English sentence:")
print(english_sentences[0])
print("\nExample Italian sentence:")
print(italian_sentences[0])
print("\nExample English question:")
print(english_test_questions[0])
print("\nExample Italian question:")
print(italian_test_questions[0])


## 2. Load the Model

We use the base Qwen model because the SAE notebook uses Qwen-Scope SAEs trained for this base-model family. A base model is not like ChatGPT or other commercial LLM products: it has been trained on very large amounts of text and is good at continuing a sequence of language. It is not specialized to follow instructions or act in an environment, as coding assistants do, for example.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto" if DEVICE == "cuda" else None,
)
model.eval()

MODEL_DEVICE = next(model.parameters()).device
ALL_LAYERS = list(range(model.config.num_hidden_layers))

print(f"Model device: {MODEL_DEVICE}")
print(f"Hidden size : {model.config.hidden_size}")
print(f"Layers      : {model.config.num_hidden_layers}")


## 3. Black-Box Control with Prompts

We start by prompting our model to control its behavior. We ask English and Italian questions, then request either English or Italian answers. This gives a baseline for language control without touching model internals.

For quick scoring, we use a small marker-word classifier based mostly on function words. It is deliberately external to the model, so we are not asking the same model to judge its own outputs.


In [ ]:
# Explicit styles are used for prompting; neutral styles are used later for steering tests.
PROMPT_STYLES = {
    "english": "Answer the question in English.",
    "italian": "Rispondi alla domanda in italiano.",
}

def build_prompt(question: str, prompt_style: str) -> str:
    if prompt_style == "italian":
        return f"{PROMPT_STYLES[prompt_style]}\n\nDomanda: {question}\nRisposta:"
    return f"{PROMPT_STYLES[prompt_style]}\n\nQuestion: {question}\nAnswer:"


def generate_text(prompt: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(MODEL_DEVICE)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print(build_prompt(english_test_questions[0], "english"), "\n")
print("-"*80)
print(build_prompt(english_test_questions[0], "italian"), "\n")
print("-"*80)
print(build_prompt(italian_test_questions[0], "english"), "\n")
print("-"*80)
print(build_prompt(italian_test_questions[0], "italian"), "\n")

In [ ]:
ENGLISH_MARKERS = {
    "the", "and", "that", "this", "these", "those", "with", "from", "for", "are", "is", "was", "were",
    "be", "been", "being", "to", "of", "in", "on", "at", "by", "as", "it", "its", "they", "them", "their",
    "we", "you", "he", "she", "or", "if", "then", "than", "but", "not", "also", "when", "while", "because",
    "about", "between", "through", "into", "over", "under", "more", "less", "very", "can", "could", "should",
    "would", "will", "may", "might", "must", "have", "has", "had", "do", "does", "did",
}

ITALIAN_MARKERS = {
    "il", "lo", "la", "i", "gli", "le", "un", "uno", "una", "di", "a", "da", "in", "con", "su", "per",
    "tra", "fra", "che", "e", "o", "ma", "se", "poi", "come", "quando", "mentre", "perché", "non", "anche",
    "più", "meno", "molto", "questo", "questa", "questi", "queste", "quello", "quella", "quelli", "quelle",
    "mi", "ti", "si", "ci", "vi", "lui", "lei", "loro", "noi", "voi", "suo", "sua", "è", "sono",
    "era", "erano", "essere", "stato", "stata", "hanno", "ha", "ho", "hai", "abbiamo", "può", "possono",
    "deve", "devono", "dovrebbe", "dovrebbero", "del", "dello", "della", "dei", "degli", "delle", "al", "allo",
    "alla", "ai", "agli", "alle", "nel", "nello", "nella", "nei", "negli", "nelle",
}


def classify_language_by_markers(text: str) -> str:
    tokens = re.findall(r"[a-zàèéìòù']+", text.lower())
    english_score = sum(token in ENGLISH_MARKERS for token in tokens)
    italian_score = sum(token in ITALIAN_MARKERS for token in tokens)

    if italian_score > english_score:
        return "italian"
    elif english_score > italian_score:
        return "english"
    else:
      return "unknown"


In [ ]:
prompt_cases = [
    ("english_questions", english_test_questions[:5], ["english", "italian"]),
    ("italian_questions", italian_test_questions[:5], ["english", "italian"]),
]

prompt_rows = []
for question_set, questions, prompt_styles in prompt_cases:
    for question in tqdm(questions, desc=f"Prompt baseline: {question_set}"):
        for prompt_style in prompt_styles:
            prompt = build_prompt(question, prompt_style)
            response = generate_text(prompt, max_new_tokens=MAX_NEW_TOKENS)
            detected_language = classify_language_by_markers(response)
            prompt_rows.append({
                "question_set": question_set,
                "question": question,
                "prompt_style": prompt_style,
                "detected_language": detected_language,
                "response": response,
            })

prompt_results = pd.DataFrame(prompt_rows)
prompt_results


In [ ]:
def language_count_table(results: pd.DataFrame, group_column: str) -> pd.DataFrame:
    counts = (
        results
        .groupby([group_column, "detected_language"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=LANGUAGE_ORDER, fill_value=0)
    )
    return counts


def plot_language_count_bars(results: pd.DataFrame, group_column: str, title: str, ax=None) -> pd.DataFrame:
    counts = language_count_table(results, group_column)
    colors = [LANGUAGE_COLORS[language] for language in counts.columns]
    ax = counts.plot(kind="bar", ax=ax, color=colors, rot=0, width=0.75)
    ax.set_title(title)
    ax.set_xlabel(group_column.replace("_", " "))
    ax.set_ylabel("Number of responses")
    ax.legend(title="Detected language", frameon=False)
    return counts

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, (question_set, group) in zip(axes, prompt_results.groupby("question_set", sort=False)):
    plot_language_count_bars(group, group_column="prompt_style", title=question_set, ax=ax)
fig.suptitle("Prompt-only language control", y=1.04)
plt.tight_layout()
plt.show()


## 4. All-Layer Residual-Stream Extraction

Now we stop treating the model as a black box. For each English/Italian sentence, we run a normal forward pass and save the residual-stream vector at every layer.

An important tool we use here is a PyTorch forward hook. A hook is a temporary function attached to a layer. When that layer runs, the hook receives the layer output. Here the hook only copies activations; later, the steering hook will edit them.


In [ ]:
def _layer_output_to_hidden_states(output: Any) -> torch.Tensor:
    """Return the hidden-state tensor when a transformer layer returns either a tensor or a tuple."""
    return output[0] if isinstance(output, tuple) else output


def capture_residual_batch(texts: list[str], layers: list[int] = ALL_LAYERS) -> dict[int, torch.Tensor]:
    captured_hidden_states: dict[int, torch.Tensor] = {}
    hook_handles = []

    def make_capture_hook(layer: int):
        def hook_fn(module, inputs, output):
            hidden_states = _layer_output_to_hidden_states(output)
            captured_hidden_states[layer] = hidden_states.detach()
        return hook_fn

    for layer in layers:
        layer_module = model.model.layers[layer]
        hook_handles.append(layer_module.register_forward_hook(make_capture_hook(layer)))

    encoded = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
    encoded = {name: value.to(MODEL_DEVICE) for name, value in encoded.items()}
    with torch.no_grad():
        model(**encoded, use_cache=False)

    for handle in hook_handles:
        handle.remove()

    last_token_positions = encoded["attention_mask"].sum(dim=1) - 1
    batch_positions = torch.arange(len(texts), device=MODEL_DEVICE)
    residuals_by_layer = {
        layer: hidden_states[batch_positions, last_token_positions].float().cpu()
        for layer, hidden_states in captured_hidden_states.items()
    }
    return residuals_by_layer


def extract_residuals_by_layer(
    texts: list[str],
    layers: list[int] = ALL_LAYERS,
    batch_size: int = BATCH_SIZE,
) -> dict[int, torch.Tensor]:
    residual_batches_by_layer = {layer: [] for layer in layers}
    for start in tqdm(range(0, len(texts), batch_size), desc="Extract residuals"):
        batch_texts = texts[start:start + batch_size]
        batch_residuals = capture_residual_batch(batch_texts, layers=layers)
        for layer in layers:
            residual_batches_by_layer[layer].append(batch_residuals[layer])
    return {
        layer: torch.cat(residual_batches, dim=0)
        for layer, residual_batches in residual_batches_by_layer.items()
    }

english_residuals_by_layer = extract_residuals_by_layer(english_sentences, layers=ALL_LAYERS)
italian_residuals_by_layer = extract_residuals_by_layer(italian_sentences, layers=ALL_LAYERS)

## 5. 2D PCA Layer Scan

For each layer, we run a 2D PCA to compress the English and Italian residual vectors to two dimensions. The model activations live in a space with thousands of dimensions. Intuitively, if two dimensions already reveal a clear separation between English and Italian examples, this suggests that language information is organized in a relatively accessible direction or subspace at that layer.

These plots do not prove that a single steering vector will work, but they give us a useful diagnostic: layers with clearer separation are plausible places to look for a contrastive direction.


In [ ]:
pca_rows = []
pca_points_by_layer = {}
language_labels = np.array(
    ["english"] * len(english_sentences) + ["italian"] * len(italian_sentences)
)

for layer in ALL_LAYERS:
    layer_residuals = torch.cat([
        english_residuals_by_layer[layer],
        italian_residuals_by_layer[layer],
    ], dim=0).numpy()
    pca = PCA(n_components=2, random_state=SEED)
    projected_residuals = pca.fit_transform(layer_residuals)
    pca_points_by_layer[layer] = projected_residuals

    pca_rows.append({
        "layer": layer,
        "pca_variance_explained": float(pca.explained_variance_ratio_.sum()),
    })

pca_summary = pd.DataFrame(pca_rows)

In [ ]:
n_layers = len(ALL_LAYERS)
n_cols = 4
n_rows = int(np.ceil(n_layers / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.1 * n_rows), squeeze=False)

for ax, layer in zip(axes.flat, ALL_LAYERS):
    projected_residuals = pca_points_by_layer[layer]
    for language, marker in [("english", "o"), ("italian", "^")]:
        mask = language_labels == language
        ax.scatter(
            projected_residuals[mask, 0],
            projected_residuals[mask, 1],
            label=language,
            marker=marker,
            alpha=0.72,
            s=24,
            color=LANGUAGE_COLORS[language],
            edgecolor="white",
            linewidth=0.25,
        )
    row = pca_summary.query("layer == @layer").iloc[0]
    ax.set_title(f"Layer {layer}  |  var={row.pca_variance_explained:.2f}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)

for ax in axes.flat[n_layers:]:
    ax.axis("off")

fig.suptitle("PCA view of English vs Italian residual-stream states", y=1.01, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Choose Layers and Extract Steering Vectors

After inspecting the PCA grid, we choose a compact group of layers. For each selected layer, we compute one contrastive direction from the residual-stream activations.

Let $h_{\ell}(x) \in \mathbb{R}^{d_{\text{model}}}$ be the residual-stream vector at layer $\ell$ for a sentence $x$. For the English and Italian sentence sets, define the two centroids:

$$
\mu^{\text{en}}_{\ell} = \frac{1}{N_{\text{en}}}\sum_{i=1}^{N_{\text{en}}} h_{\ell}(x^{\text{en}}_i)
$$

$$
\mu^{\text{it}}_{\ell} = \frac{1}{N_{\text{it}}}\sum_{i=1}^{N_{\text{it}}} h_{\ell}(x^{\text{it}}_i)
$$

The steering vector is the difference between these two centroids:

$$
v^{\text{it-en}}_{\ell} = \mu^{\text{it}}_{\ell} - \mu^{\text{en}}_{\ell}
$$

The intuition is that averaging over many sentences cancels out idiosyncratic details of individual examples, while preserving the systematic difference between the two groups. If the PCA plot showed English and Italian examples forming separated clouds at layer $\ell$, then $v^{\text{it-en}}_{\ell}$ is the direction connecting the English centroid to the Italian centroid in the original high-dimensional residual space.

Before using it, we normalize the vector:

$$
\hat{v}^{\text{it-en}}_{\ell} = \frac{v^{\text{it-en}}_{\ell}}{\lVert v^{\text{it-en}}_{\ell} \rVert_2}
$$

During generation, the intervention adds this direction to the residual stream:

$$
h'_{\ell,t} = h_{\ell,t} + \alpha\,\hat{v}^{\text{it-en}}_{\ell}
$$

where $t$ indexes the token position. Positive $\alpha$ moves activations toward the Italian centroid; negative $\alpha$ moves them in the opposite direction, toward the English side.


In [ ]:
SELECTED_STEERING_LAYERS = [4]
PRIMARY_STEERING_LAYER = 4

steering_vectors_by_layer = {}

for layer in SELECTED_STEERING_LAYERS:
    italian_centroid = italian_residuals_by_layer[layer].mean(dim=0)
    english_centroid = english_residuals_by_layer[layer].mean(dim=0)
    steering_vector = italian_centroid - english_centroid
    steering_vector = steering_vector / steering_vector.norm()
    steering_vectors_by_layer[layer] = steering_vector

print(f"\nPrimary steering layer for quick demos: {PRIMARY_STEERING_LAYER}")


In [ ]:
def make_steering_hook(vector: torch.Tensor, alpha: float):
    steering_direction = vector.to(MODEL_DEVICE, dtype=dtype)

    def hook_fn(module, inputs, output):
        hidden_states = _layer_output_to_hidden_states(output)
        delta = alpha * steering_direction.view(1, 1, -1)
        steered_hidden_states = hidden_states + delta
        if isinstance(output, tuple):
            return (steered_hidden_states, *output[1:])
        return steered_hidden_states

    return hook_fn


def generate_with_steering(
    question: str,
    prompt_style: str,
    layer: int,
    alpha: float,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> str:
    prompt = build_prompt(question, prompt_style)
    inputs = tokenizer(prompt, return_tensors="pt").to(MODEL_DEVICE)
    steering_vector = steering_vectors_by_layer[layer]
    hook = make_steering_hook(steering_vector, alpha)
    handle = model.model.layers[layer].register_forward_hook(hook)
    try:
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        handle.remove()
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

quick_demo_cases = [
    {"question_set": "english_questions", "prompt_style": "english", "question": english_test_questions[0]},
    {"question_set": "italian_questions", "prompt_style": "italian", "question": italian_test_questions[0]},
]

quick_demo_rows = []
for case in quick_demo_cases:
    for alpha in [-20.0, 0.0, 20.0]:
        response = generate_with_steering(
            case["question"],
            prompt_style=case["prompt_style"],
            layer=PRIMARY_STEERING_LAYER,
            alpha=alpha,
        )
        quick_demo_rows.append({
            "question_set": case["question_set"],
            "prompt_style": case["prompt_style"],
            "question": case["question"],
            "layer": PRIMARY_STEERING_LAYER,
            "alpha": alpha,
            "detected_language": classify_language_by_markers(response),
            "response": response,
        })

pd.DataFrame(quick_demo_rows)


## 7. Steering Sweep Across Layers, Alphas, and Questions

Now we repeat the intervention systematically across selected layers, steering strengths, and multiple questions. We use neutral prompts that ask the model to answer a question without explicitly requesting English or Italian.

The vector is `mean(Italian) - mean(English)`: positive alpha should push toward Italian, while negative alpha subtracts that direction and should push toward English.


In [ ]:
ALPHAS = [-20, 0, 20]
SWEEP_CASES = [
    {"question_set": "english_questions", "prompt_style": "english", "questions": english_test_questions[:5]},
    {"question_set": "italian_questions", "prompt_style": "italian", "questions": italian_test_questions[:5]},
]

sweep_rows = []
for case in SWEEP_CASES:
    for layer in tqdm(SELECTED_STEERING_LAYERS, desc=f"Layers: {case['question_set']}"):
        for alpha in tqdm(ALPHAS, desc=f"Alphas L{layer}", leave=False):
            for question in case["questions"]:
                response = generate_with_steering(
                    question,
                    prompt_style=case["prompt_style"],
                    layer=layer,
                    alpha=float(alpha),
                    max_new_tokens=MAX_NEW_TOKENS,
                )
                detected_language = classify_language_by_markers(response)
                sweep_rows.append({
                    "question_set": case["question_set"],
                    "prompt_style": case["prompt_style"],
                    "question": question,
                    "layer": layer,
                    "alpha": alpha,
                    "detected_language": detected_language,
                    "response": response,
                })

steering_sweep = pd.DataFrame(sweep_rows)
steering_sweep


In [ ]:
def language_balance(group: pd.DataFrame) -> float:
    english_count = (group["detected_language"] == "english").sum()
    italian_count = (group["detected_language"] == "italian").sum()
    detected_count = english_count + italian_count
    if detected_count == 0:
        return 0.0
    return (italian_count - english_count) / detected_count

steering_summary = (
    steering_sweep
    .groupby(["question_set", "layer", "alpha", "detected_language"], as_index=False)
    .size()
    .rename(columns={"size": "n_responses"})
)

balance_rows = []
for (question_set, layer, alpha), group in steering_sweep.groupby(
    ["question_set", "layer", "alpha"]
):
    balance_rows.append({
        "question_set": question_set,
        "layer": layer,
        "alpha": alpha,
        "language_balance": language_balance(group),
    })
balance = pd.DataFrame(balance_rows)

question_sets = balance["question_set"].drop_duplicates().tolist()
layers = sorted(balance["layer"].unique())

fig, axes = plt.subplots(
    len(layers),
    len(question_sets),
    figsize=(5.5 * len(question_sets), 5.5 * len(layers)),
    sharex=True,
    sharey=True,
    squeeze=False,
)

for row, layer in enumerate(layers):
    for column, question_set in enumerate(question_sets):
        ax = axes[row, column]
        plot_data = (
            balance.query("question_set == @question_set and layer == @layer")
            .sort_values("alpha")
        )
        bar_colors = [
            LANGUAGE_COLORS["italian"] if value >= 0 else LANGUAGE_COLORS["english"]
            for value in plot_data["language_balance"]
        ]
        bars = ax.bar(
            plot_data["alpha"].astype(str),
            plot_data["language_balance"],
            color=bar_colors,
            width=0.7,
        )
        ax.axhline(0, color="black", linewidth=1)
        ax.set_ylim(-1.08, 1.08)
        ax.set_title(f"Layer {layer}: {question_set}")
        ax.bar_label(bars, fmt="%+.2f", padding=3, fontsize=9)

        if row == len(layers) - 1:
            ax.set_xlabel("Intervention strength alpha")
        if column == 0:
            ax.set_ylabel("Italian share - English share")

fig.suptitle("Language balance under steering", y=1.01, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## Summary

Prompting changed the output language from the outside. PCA showed that English and Italian sentences occupy different regions of the residual stream at some layers. Steering then changed generation by adding a learned internal direction during the forward pass.

The key object is the contrastive vector `mean(Italian) - mean(English)`. Positive alpha adds the Italian direction; negative alpha subtracts it. Notebook 02 keeps the same language-control task, but replaces one dense direction with sparse SAE features that can be inspected one by one.
